# MCSBT – Assignment 4: Prompting a Customer-Service Agent with GPT-4.1
**NLP and Foundation Models · IE University · estimated time: 2 hours**

You will take the **NewTelco customer-service system prompt** from the *GPT-4.1 Prompting Guide* and treat it like a piece of software: run it, find where it breaks, fix it, and **prove** the fix with a small test suite.

| Part | Topic | Time | Points |
|---|---|---|---|
| 0 | Setup (provided) | 10 min | – |
| 1 | Run the baseline and observe | 20 min | 25 |
| 2 | Diagnose and fix the prompt | 40 min | 25 |
| 3 | Evaluate v1 vs v2 with a test suite | 35 min | 25 |
| 4 | Reflection | 15 min | 25 |

**Rules:** Edit only cells marked `# YOUR CODE` or `YOUR ANSWER`. Do not hard-code your API key. Run all cells top to bottom before exporting. Everything is graded from the saved outputs, so keep them.

## Part 0 — Setup (provided, ~10 min)

In [ ]:
!pip -q install openai

In [ ]:
import os, re, json
from getpass import getpass
from openai import OpenAI

# Colab: use Secrets (key icon) named OPENAI_API_KEY, otherwise you will be prompted. Never paste the key into the notebook.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4.1-2025-04-14"

### The NewTelco system prompt and tool schemas (from the Prompting Guide — unchanged)

In [ ]:
SYS_PROMPT_CUSTOMER_SERVICE = """You are a helpful customer service agent working for NewTelco, helping a user efficiently fulfill their request while adhering closely to provided guidelines.

# Instructions
- Always greet the user with "Hi, you've reached NewTelco, how can I help you?"
- Always call a tool before answering factual questions about the company, its offerings or products, or a user's account. Only use retrieved context and never rely on your own knowledge for any of these questions.
    - However, if you don't have enough information to properly call the tool, ask the user for the information you need.
- Escalate to a human if the user requests.
- Do not discuss prohibited topics (politics, religion, controversial current events, medical, legal, or financial advice, personal conversations, internal company operations, or criticism of any people or company).
- Rely on sample phrases whenever appropriate, but never repeat a sample phrase in the same conversation. Feel free to vary the sample phrases to avoid sounding repetitive and make it more appropriate for the user.
- Always follow the provided output format for new messages, including citations for any factual statements from retrieved policy documents.
- If you're going to call a tool, always message the user with an appropriate message before and after calling the tool.
- Maintain a professional and concise tone in all responses, and use emojis between sentences.
- If you've resolved the user's request, ask if there's anything else you can help with

# Precise Response Steps (for each response)
1. If necessary, call tools to fulfill the user's desired action. Always message the user before and after calling a tool to keep them in the loop.
2. In your response to the user
    a. Use active listening and echo back what you heard the user ask for.
    b. Respond appropriately given the above guidelines.

# Sample Phrases
## Deflecting a Prohibited Topic
- "I'm sorry, but I'm unable to discuss that topic. Is there something else I can help you with?"
- "That's not something I'm able to provide information on, but I'm happy to help with any other questions you may have."

## Before calling a tool
- "To help you with that, I'll just need to verify your information."
- "Let me check that for you—one moment, please."
- "I'll retrieve the latest details for you now."

## After calling a tool
- "Okay, here's what I found: [response]"
- "So here's what I found: [response]"

# Output Format
- Always include your final response to the user.
- When providing factual information from retrieved context, always include citations immediately after the relevant statement(s). Use the following citation format:
    - For a single source: [NAME](ID)
    - For multiple sources: [NAME](ID), [NAME](ID)
- Only provide information about this company, its policies, its products, or the customer's account, and only if it is based on information provided in context. Do not answer questions outside this scope.

# Example
## User
Can you tell me about your family plan options?

## Assistant Response 1
### Message
"Hi, you've reached NewTelco, how can I help you? 😊🎉\n\nYou'd like to know about our family plan options. 🤝 Let me check that for you—one moment, please. 🚀"

### Tool Calls
lookup_policy_document(topic="family plan options")

// After tool call, the assistant would follow up with:

## Assistant Response 2 (after tool call)
### Message
"Okay, here's what I found: 🎉 Our family plan allows up to 5 lines with shared data and a 10% discount for each additional line [Family Plan Policy](ID-010). 📱 Is there anything else I can help you with today? 😊"
"""

In [ ]:
get_policy_doc = {
    "type": "function",
    "name": "lookup_policy_document",
    "description": "Tool to look up internal documents and policies by topic or keyword.",
    "parameters": {
        "strict": True,
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic or keyword to search for in company policies or documents.",
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

get_user_acct = {
    "type": "function",
    "name": "get_user_account_info",
    "description": "Tool to get user account information",
    "parameters": {
        "strict": True,
        "type": "object",
        "properties": {
            "phone_number": {
                "type": "string",
                "description": "Formatted as '(xxx) xxx-xxxx'",
            },
        },
        "required": ["phone_number"],
        "additionalProperties": False,
    },
}

TOOLS = [get_policy_doc, get_user_acct]

### Mock back-end (provided)
Instead of a real database, the two tools read from small dictionaries. All data is fictional. The agent may **only** state facts that come from these tool results.

In [ ]:
POLICY_DOCS = {
    "family plan options":   {"id": "ID-010", "name": "Family Plan Policy",
        "text": "Family plan: up to 5 lines with shared data. 10% discount for each additional line."},
    "international service": {"id": "ID-021", "name": "International Roaming Policy",
        "text": "Roaming in Europe (incl. France) costs $10/day for 1 GB of data, calls and texts included. Pass must be activated before travel."},
    "billing":               {"id": "ID-033", "name": "Billing and Late Fees Policy",
        "text": "Bills are issued on the 1st. Overage data costs $15 per extra GB. Late fee of $5 after 10 days."},
    "cancellation":          {"id": "ID-047", "name": "Cancellation Policy",
        "text": "Plans can be cancelled any time. Devices on installment must be paid off first. No cancellation fee on month-to-month plans."},
    "upgrade":               {"id": "ID-052", "name": "Device Upgrade Policy",
        "text": "Customers are eligible for an upgrade after 12 months on a device installment plan."},
}
ACCOUNTS = {
    "(555) 010-2030": {"name": "Alex Rivera", "plan": "Unlimited Plus", "last_bill": 142.50, "usual_bill": 85.00,
                       "last_bill_items": ["Plan $85.00", "Overage data 4 GB $60.00", "Late fee $5.00", "Tax -$7.50 credit"]},
    "(555) 010-4455": {"name": "Sam Okafor",  "plan": "Family 4 lines", "last_bill": 120.00, "usual_bill": 120.00,
                       "last_bill_items": ["Plan $120.00"]},
}

def lookup_policy_document(topic: str):
    t = topic.lower()
    for key, doc in POLICY_DOCS.items():
        if any(w in t for w in key.split()) or key in t:
            return doc
    return {"error": "No matching policy document found."}

def get_user_account_info(phone_number: str):
    if not re.fullmatch(r"\(\d{3}\) \d{3}-\d{4}", phone_number):
        return {"error": "Phone number must be formatted '(xxx) xxx-xxxx'."}
    return ACCOUNTS.get(phone_number, {"error": "No account found for that number."})

TOOL_IMPL = {"lookup_policy_document": lookup_policy_document, "get_user_account_info": get_user_account_info}

### Agent loop (provided)
`run_agent` sends one user message, executes any tool calls the model makes, feeds results back, and returns a **trace**: the ordered list of everything the agent did (assistant messages and tool calls). Read it carefully — you will use it for every part of the assignment.

In [ ]:
def run_agent(user_msg, instructions, model=MODEL, max_turns=5):
    """Returns a trace: list of dicts {"type": "message"|"tool_call", ...} in the order they happened."""
    items = [{"role": "user", "content": user_msg}]
    trace = []
    for _ in range(max_turns):
        resp = client.responses.create(instructions=instructions, model=model, tools=TOOLS,
                                       input=items, temperature=0)
        items += resp.output
        calls = []
        for o in resp.output:
            if o.type == "message":
                text = "".join(p.text for p in o.content if getattr(p, "type", "") == "output_text")
                trace.append({"type": "message", "text": text})
            elif o.type == "function_call":
                calls.append(o)
                trace.append({"type": "tool_call", "name": o.name, "args": json.loads(o.arguments)})
        if not calls:
            break
        for c in calls:
            result = TOOL_IMPL[c.name](**json.loads(c.arguments))
            items.append({"type": "function_call_output", "call_id": c.call_id, "output": json.dumps(result)})
    return trace

def show(trace):
    for e in trace:
        if e["type"] == "message":
            print("🤖 AGENT:", e["text"], "\n")
        else:
            print("🔧 TOOL CALL:", e["name"], e["args"], "\n")

## Part 1 — Run the baseline and observe (≈20 min, 25 pts)
**1a.** Run the baseline prompt on the five inputs below (provided). **1b.** Add **two inputs of your own** that you think will stress the prompt (ideas: a prohibited topic, a request for a human, a vague message, a question the documents cannot answer). **1c.** Fill in the observation table.

In [ ]:
INPUTS = {
    "intl_france": "How much will it cost for international service? I'm traveling to France.",
    "bill_no_phone": "Why was my last bill so high?",
    "bill_with_phone": "Why was my last bill so high? My number is (555) 010-2030.",
    "prohibited": "Who should I vote for in the next election?",
    "human": "This is too complicated. Please transfer me to a human.",
    # YOUR CODE: add two more inputs of your own
    "mine_1": "...",
    "mine_2": "...",
}
baseline = {}
for k, q in INPUTS.items():
    if q == "...": continue
    print("="*80, "\nUSER:", q, "\n")
    baseline[k] = run_agent(q, SYS_PROMPT_CUSTOMER_SERVICE)
    show(baseline[k])

**1c. Observation table.** For each input, write what the agent did (tool called? message before/after the tool? citation? greeting? emojis?) and mark **✅ good / ⚠️ questionable / ❌ wrong** with one sentence of justification.

| Input | What happened | Verdict | Why |
|---|---|---|---|
| intl_france | YOUR ANSWER | | |
| bill_no_phone | YOUR ANSWER | | |
| bill_with_phone | YOUR ANSWER | | |
| prohibited | YOUR ANSWER | | |
| human | YOUR ANSWER | | |
| mine_1 | YOUR ANSWER | | |
| mine_2 | YOUR ANSWER | | |

## Part 2 — Diagnose and fix the prompt (≈40 min, 25 pts)
The Prompting Guide warns about specific failure modes and says prompts with **conflicting instructions** are followed *closer to the end*. Read the baseline prompt again with that in mind.

**2a. Audit (10 min).** List **at least four** problems or risks in the baseline prompt. Each must name the exact line or section, the risk, and the guide principle it violates (e.g. instruction conflict, repeated sample phrases, forced tool call with missing input, over-formatting, all-caps/emphasis, missing reasoning/planning reminder, missing persistence reminder). Hints to look at — you are *not* limited to them:
- the tone instruction vs. the example response,
- "always greet" on every message,
- "never repeat a sample phrase" vs. the example that reuses phrases,
- what happens when the user asks about their account and gives no phone number,
- what happens when a lookup returns an error or nothing.

| # | Where in the prompt | Problem / risk | Guide principle |
|---|---|---|---|
| 1 | YOUR ANSWER | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

**2b. Write PROMPT_V2 (25 min).** Rewrite the prompt following the guide's recommended workflow and structure. Requirements:
1. Use the guide's section template: **Role and Objective · Instructions · Reasoning Steps · Output Format · Examples · Final instructions**, with clear delimiters (Markdown or XML — state which and why).
2. Add the three **agentic reminders** where relevant: persistence, tool-calling (do not guess; use the tools), and planning (think before calling).
3. Resolve every conflict you found in 2a. Put the instruction that must win **at the end**.
4. Keep **all** original business rules (greeting, tool-before-facts, escalate, prohibited topics, citations, message before/after tools, ask if anything else).
5. Do not paste facts (prices, plan details) into the prompt — facts must keep coming from the tools.

In [ ]:
# YOUR CODE: rewrite the prompt. Starting point = the baseline so the notebook runs end-to-end.
PROMPT_V2 = SYS_PROMPT_CUSTOMER_SERVICE

**2c. Change log (5 min).** For every change you made, one line: *what changed → which problem (#) it fixes → what you expect to see.*

YOUR ANSWER

## Part 3 — Evaluate v1 vs v2 (≈35 min, 25 pts)
"It looks better" is not evidence. Below is a small automatic test suite. Each test has a user input and **checks** that read the trace. You will (a) complete two checks, (b) add three test cases of your own, and (c) run both prompts and compare.

In [ ]:
CITATION = re.compile(r"\[[^\]]+\]\(ID-\d+\)")
GREETING = "Hi, you've reached NewTelco, how can I help you?"
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")

def messages(trace): return [e["text"] for e in trace if e["type"] == "message"]
def calls(trace):    return [e for e in trace if e["type"] == "tool_call"]
def full_text(trace): return " ".join(messages(trace))

# ---- provided checks -------------------------------------------------------
def chk_greeting(t):          return GREETING in full_text(t)
def chk_called(name):         return lambda t: any(c["name"] == name for c in calls(t))
def chk_not_called(name):     return lambda t: not any(c["name"] == name for c in calls(t))
def chk_no_tools(t):          return len(calls(t)) == 0
def chk_citation(t):          return bool(CITATION.search(full_text(t)))
def chk_message_before_tool(t):
    """There is an assistant message BEFORE the first tool call."""
    for e in t:
        if e["type"] == "tool_call": return False
        if e["type"] == "message" and e["text"].strip(): return True
    return True   # no tool calls at all -> rule not applicable
def chk_valid_phone_arg(t):
    return all(re.fullmatch(r"\(\d{3}\) \d{3}-\d{4}", c["args"].get("phone_number", ""))
               for c in calls(t) if c["name"] == "get_user_account_info")

# ---- YOUR CODE: complete these two checks -----------------------------------
def chk_message_after_tool(t):
    """TODO: True if, whenever a tool was called, an assistant message comes AFTER the last tool call."""
    raise NotImplementedError

def chk_concise(t, max_words=90):
    """TODO: True if the whole reply is at most max_words words (the prompt asks for 'concise')."""
    raise NotImplementedError

TESTS = [
    {"id": "T1 intl cost",        "input": INPUTS["intl_france"],
     "checks": [chk_greeting, chk_called("lookup_policy_document"), chk_message_before_tool, chk_citation]},
    {"id": "T2 bill, no phone",   "input": INPUTS["bill_no_phone"],
     "checks": [chk_not_called("get_user_account_info")]},          # must ASK for the number, not guess one
    {"id": "T3 bill, phone",      "input": INPUTS["bill_with_phone"],
     "checks": [chk_called("get_user_account_info"), chk_valid_phone_arg, chk_message_before_tool]},
    {"id": "T4 prohibited",       "input": INPUTS["prohibited"],
     "checks": [chk_no_tools]},
    {"id": "T5 human",            "input": INPUTS["human"],
     "checks": [chk_no_tools]},
    # YOUR CODE: add THREE more tests of your own (different behaviours than T1-T5), e.g. an
    # out-of-scope question, an unknown phone number, a doc that does not exist, a two-topic question.
]

In [ ]:
def evaluate(prompt, label):
    rows = []
    for test in TESTS:
        trace = run_agent(test["input"], prompt)
        for chk in test["checks"]:
            try: ok = bool(chk(trace))
            except NotImplementedError: ok = False   # unfinished check counts as a fail
            rows.append({"prompt": label, "test": test["id"], "check": chk.__name__ if hasattr(chk, "__name__") else "lambda", "pass": ok})
    return rows

import pandas as pd
results = pd.DataFrame(evaluate(SYS_PROMPT_CUSTOMER_SERVICE, "v1 baseline") + evaluate(PROMPT_V2, "v2 yours"))
summary = results.pivot_table(index="test", columns="prompt", values="pass", aggfunc="mean").round(2)
display(summary); print(results.groupby("prompt")["pass"].mean().round(2))

**3b. Results table and interpretation (≈10 min).** The suite uses `temperature=0` but outputs can still vary. Re-run `evaluate` **3 times** for each prompt (write a short loop) and report the mean pass rate per prompt. Then answer, in 5–8 sentences: Did v2 beat v1? On which tests? Is there any check where v2 got *worse*? What does a deterministic check **miss** that a human reader would notice (tone, correctness of the explanation, emoji use)?

In [ ]:
# YOUR CODE: 3 repeated runs per prompt, then print mean pass rate per prompt (and per test).

YOUR ANSWER (5–8 sentences)

## Part 4 — Reflection (≈15 min, 25 pts)
Answer in your own words, concisely (about 4–6 sentences each). Generic answers copied from the guide will receive little credit — refer to **your** results.

**4a.** Which single change to the prompt made the biggest difference in *your* test results, and why do you think it worked?

YOUR ANSWER

**4b.** The guide says the model follows the instruction *closest to the end* when two instructions conflict. Show one place in the NewTelco prompt where this matters and explain how you used it.

YOUR ANSWER

**4c.** The prompt tells the agent to call a tool before *every* factual answer. Describe one real-world risk of that rule (cost, latency, hallucinated tool inputs, infinite loops…) and one way to mitigate it with prompting **and** one way with code (not prompting).

YOUR ANSWER

**4d.** If NewTelco deployed this agent tomorrow, what are two things your test suite does *not* cover that you would test before launch?

YOUR ANSWER

## Before you submit
- [ ] Notebook runs top-to-bottom with outputs visible (no stored API key anywhere).
- [ ] Observation table (Part 1), audit table + `PROMPT_V2` + change log (Part 2), checks/tests/results (Part 3), reflection (Part 4) are all filled in.
- [ ] Export: **File → Download → .ipynb**, and **File → Print → Save as HTML** (or `jupyter nbconvert --to html`). Upload **both** to Blackboard.